# Step 50 Run LOSO K-Sweep Model Selection

## What This Notebook Does

This is the public Stage-5 screening notebook for fusion-HMM model-order selection.

It preserves the original broad K-sweep workflow that:
- runs leave-one-subject-out cross-validation over `K = 2..12`,
- tracks held-out free energy and feasibility across seeds,
- writes the main K-sweep summary tables,
- and records the 1-SE rule plus the local-minimum screening outputs.
- This stage consumes the canonical Stage-4 alignment branch: `intermediate`, `nolags`, `minlen15`, where `intermediate` denotes the manuscript’s middle-ground TR-retention policy between earlier lenient and strict variants.

## When To Run It

Run this notebook after the Stage-4 canonical segment dataset has been built.

Run this notebook before:
- `step51_run_loso_shortlist_stability_checks.ipynb`
- `step52_build_figure2_and_table_s8_model_selection_summary.ipynb`

The cleaned public default is the frozen manuscript dataset:
- `FEATURE_MODE = "nolags"`
- `MINLEN = 15`

## Manuscript Linkage

- Main Methods 2.5.1
- Supplementary Methods 1.6
- Figure 2 support
- Table S8 support

## Inputs Expected

- Stage-4 final segment root containing the canonical `intermediate/` dataset branch
- `segments_manifest.tsv` and the segment `.npy` files it references

## Outputs Written

- `cv_results.tsv`
- `cv_candidates_long.tsv`
- `fold_meta.tsv`
- `summary_byK_selected.tsv`
- `summary_byK_candidates.tsv`
- `paired_tests_vs_bestK.tsv`
- `K_selection_recommendation.json`
- free-energy and feasibility plots

## Environment Notes

This stage depends on:
- `tensorflow`
- `osl_dynamics`
- GPU-sensitive execution for the main workflow

The preserved source notebook still uses XLA-off settings, memory-growth or GPU-cap handling, and chunk/resume logic for long runs. CPU-only execution is possible in code, but it is likely slower and less practical for the full sweep.

## Interpretation Note

This notebook is the broad screening stage only. Higher-K solutions, including `K=12`, can remain visible after screening. The final manuscript choice of `K=3` was not treated as coming from one fully automatic rule alone. It combined these screening outputs with the later shortlist stability and interpretability check.


In [ ]:
# Step 0. Import Python modules and locate the Stage-5 helper file

import sys
from pathlib import Path

STAGE5_DIR = Path.cwd()
if not (STAGE5_DIR / "stage5_hmm_selection_helpers.py").exists():
    candidate = Path.cwd() / "notebooks" / "5_hmm_selection"
    if candidate.exists():
        STAGE5_DIR = candidate

if str(STAGE5_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE5_DIR.resolve()))

from stage5_hmm_selection_helpers import run_wrapped_k_sweep_source


In [ ]:
# Step 1. User-editable roots and preserved run settings
#
# Set SEGMENTS_ROOT to the Stage-4 final output root that contains the
# `intermediate/` subfolder built in Stage 4.

SEGMENTS_ROOT = Path("<SET_SEGMENTS_ROOT>")
MODEL_SELECTION_ROOT = Path("<SET_MODEL_SELECTION_ROOT>")

DATA_VARIANT = "intermediate"
FEATURE_MODE = "nolags"
MINLEN = 15

# Optional explicit manifest override.
# Leave this as None to use the standard Stage-4 manifest location.
MANIFEST_TSV = None

# Broad screening grid for the manuscript model-order sweep.
K_GRID = list(range(2, 13))

# Chunk/resume and debug controls for long runs.
MAX_NEW_PAIRS_PER_RUN = 15
DEBUG_MAX_FOLDS = None
DEBUG_K_GRID = None
DEBUG_SEEDS = None

# Set to None to rely on TensorFlow memory-growth instead of a hard cap.
GPU_MEMORY_LIMIT_MB = 4096

FINAL_ROOT = SEGMENTS_ROOT / DATA_VARIANT
KSWEEP_OUTPUT_DIR = MODEL_SELECTION_ROOT / f"{DATA_VARIANT}_{FEATURE_MODE}_minlen{MINLEN}"
SOURCE_NOTEBOOK = STAGE5_DIR / "R01_PipelineC2_Ksweep_LOSO_OOMFIX_steps_XLAoff_chunkresume_v2REBATCH.ipynb"


In [ ]:
# Step 2. Validate the configured inputs and print a short run summary

def assert_configured_path(path_value, label, must_exist=False):
    path_str = str(path_value)
    if "<SET_" in path_str:
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")
    if must_exist and not Path(path_value).exists():
        raise FileNotFoundError(f"{label} does not exist: {path_value}")

assert_configured_path(SEGMENTS_ROOT, "SEGMENTS_ROOT", must_exist=True)
assert_configured_path(MODEL_SELECTION_ROOT, "MODEL_SELECTION_ROOT")

if not SOURCE_NOTEBOOK.exists():
    raise FileNotFoundError(f"Preserved source notebook not found: {SOURCE_NOTEBOOK}")

print("Stage 5 / Step 50: broad LOSO K-sweep model selection")
print(f"  Stage-4 segment root:  {FINAL_ROOT}")
print(f"  Output root:           {KSWEEP_OUTPUT_DIR}")
print(f"  Feature mode:          {FEATURE_MODE}")
print(f"  Minimum segment len:   {MINLEN} TR")
print(f"  K grid:                {K_GRID}")
print(f"  GPU memory cap (MB):   {GPU_MEMORY_LIMIT_MB}")
print("  Packages needed:       tensorflow, osl_dynamics, numpy, pandas, matplotlib, scipy")


In [ ]:
# Step 3. Run the preserved broad K-sweep notebook through the public wrapper

run_wrapped_k_sweep_source(
    source_notebook=SOURCE_NOTEBOOK,
    final_root=FINAL_ROOT,
    out_root=KSWEEP_OUTPUT_DIR,
    data_variant=DATA_VARIANT,
    feature_mode=FEATURE_MODE,
    minlen=MINLEN,
    manifest_tsv=MANIFEST_TSV,
    k_grid=K_GRID,
    max_new_pairs_per_run=MAX_NEW_PAIRS_PER_RUN,
    gpu_memory_limit_mb=GPU_MEMORY_LIMIT_MB,
    debug_max_folds=DEBUG_MAX_FOLDS,
    debug_k_grid=DEBUG_K_GRID,
    debug_seeds=DEBUG_SEEDS,
)


## Notes

- This notebook intentionally keeps the broad screening stage separate from the shortlist stability check in `step51_run_loso_shortlist_stability_checks.ipynb`.
- Older lagged and `minlen10` provenance paths remain preserved in archive notebooks, but they are not the public default here.
